# INSTRUCTIONS

In [ ]:
# PLEASE REFER TO README DOCUMENT BEFORE RUNNING SCRIPT

# IMPORT PACKAGES

In [ ]:
import rosbag
import rospy
import numpy as np
import pandas as pd 
from datetime import datetime
import matplotlib.pyplot as plt
import mpld3
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# USER INPUT SECTION

In [ ]:
# Use this section for 3 vehicle test run
three_vehicle_bool = False

# Input the file paths to the three bag files
# lead_v1 is the leader vehicle (vehicle 1)
# follower_v2 is the first follower vehicle (vehicle 2)
# follower_v3 is the second follower vehicle (vehicle 3)

file_read_location = "" # Input folder path where bag files are stored, ending with "/"
lead_v1_file_read = "" # Input file name here
follower_v2_file_read = "" # Input file name here
follower_v3_file_read = "" # Input file name here

# Input file path where output graphs should be saved to, ending with "/"
file_save = "" # Input here

# Input vehicle names for formatting graphs
lead_v1_name = "" # Input here
follower_v2_name = "" # Input here
follower_v3_name = "" # Input here
test_number = "" # Input test number in format "T##"
run_number = "" # Input run number in format "R#"

if lead_v1_file_read != "":
    try:
        lead_v1_bag = rosbag.Bag(file_read_location + lead_v1_file_read)
    except FileNotFoundError:
        print(f"Error: the file '{lead_v1_file_read}' was not found.")
else:
    print("No lead_v1 file entered.")
if follower_v2_file_read != "":
    try:
        follower_v2_bag = rosbag.Bag(file_read_location + follower_v2_file_read)
    except FileNotFoundError:
        print(f"Error: the file '{follower_v2_file_read}' was not found.")
else:
    print("No follower_v2 file entered.")
if follower_v3_file_read != "":
    try:
        follower_v3_bag= rosbag.Bag(file_read_location + follower_v3_file_read) 
    except FileNotFoundError:
        print(f"Error: the file '{follower_v3_file_read}' was not found.")
else:
    print("No follower_v3 file entered.")
if lead_v1_file_read != "" and follower_v2_file_read != "" and follower_v3_file_read != "" :
    three_vehicle_bool = True

In [ ]:
# Use this section for 2 vehicle test run
two_vehicle_bool = False

# Input the file paths to the two bag files
# lead_v1b is the leader vehicle (vehicle 1)
# follower_v2b is the first follower vehicle (vehicle 2)

file_b_read_location = "" # Input folder path where bag files are stored, ending with "/"
lead_v1b_file_read = "" # Input file name here
follower_v2b_file_read = "" # Input file name here

# Input file path where output graphs should be saved to, ending with "/"
file_b_save = "" # Input here

# Input vehicle names for formatting graphs
lead_v1b_name = "" # Input here
follower_v2b_name = "" # Input here
test_number_b = "" # Input test number in format "T##"
run_number_b = "" # Input run number in format "R#"

if lead_v1b_file_read != "":
    try:
        lead_v1b_bag = rosbag.Bag(file_b_read_location + lead_v1b_file_read)
    except FileNotFoundError:
        print(f"Error: the file '{lead_v1b_file_read}' was not found.")
else:
    print("No lead_v1b file entered.")
if follower_v2b_file_read != "":
    try:
        follower_v2b_bag = rosbag.Bag(file_b_read_location + follower_v2b_file_read)
    except FileNotFoundError:
        print(f"Error: the file '{follower_v2b_file_read}' was not found.")
else:
    print("No follower_v2b file entered.")
if lead_v1b_file_read != "" and follower_v2b_file_read != "":
    two_vehicle_bool = True

# DO NOT CHANGE ANYTHING BELOW THIS LINE

In [ ]:
# 3 vehicle test run
if three_vehicle_bool:

      # extract timestamp for when CARMA platform is engaged
      # msg.state == 4 indicates that the CARMA platform system is engaged
      
      lead_v1_engaged_time = 0
      for topic, msg, t in lead_v1_bag.read_messages(topics = ["/guidance/state"]):
            if msg.state == 4:
                  lead_v1_engaged_time = t
                  break
      follower_v2_engaged_time = 0
      for topic, msg, t in follower_v2_bag.read_messages(topics = ["/guidance/state"]):
            if msg.state == 4:
                  follower_v2_engaged_time = t
                  break
      follower_v3_engaged_time = 0
      for topic, msg, t in follower_v3_bag.read_messages(topics = ["/guidance/state"]):
            if msg.state == 4:
                  follower_v3_engaged_time = t
                  break


In [ ]:
# 3 vehicle test run
if three_vehicle_bool:

      # delay time until start platoon flag - lead vehicle platoon state = 4; following vehicle platoon state = 5
      
      follower_v3_platoon_flag = 0
      for topic, msg, t in follower_v3_bag.read_messages(topics = ["/guidance/platoon_info"]):
            if msg.state == 5:
                  follower_v3_platoon_flag = t
                  break

      follower_v2_platoon_flag = 0
      for topic, msg, t in follower_v2_bag.read_messages(topics = ["/guidance/platoon_info"]):
            if msg.state == 5:
                  follower_v2_platoon_flag = t
                  break

      lead_v1_platoon_flag = 0
      for topic, msg, t in lead_v1_bag.read_messages(topics = ["/guidance/platoon_info"]):
            if msg.state == 4:
                  lead_v1_platoon_flag = t
                  break

In [ ]:
# 3 vehicle test run
if three_vehicle_bool:

    # call for follower_v3 vehicle speed
    follower_v3_VehSpeed_dict = {}
    for topic, msg, t in follower_v3_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=follower_v3_engaged_time):
        follower_v3_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x  # .twist.linear.x is the vehicle's longitudinal velocity in m/s
    follower_v3_VehSpeed_df = pd.DataFrame.from_dict(follower_v3_VehSpeed_dict, orient = 'index').rename(columns = {0:'follower_v3_VehSpeed'}).rename_axis("Time").reset_index()
    follower_v3_VehSpeed_df['DateTime'] = pd.to_datetime(follower_v3_VehSpeed_df['Time'],unit='s')

    # call for follower_v2  vehicle speed
    follower_v2_VehSpeed_dict = {}
    for topic, msg, t in follower_v2_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=follower_v2_engaged_time):
        follower_v2_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x  # .twist.linear.x is the vehicle's longitudinal velocity in m/s
    follower_v2_VehSpeed_df = pd.DataFrame.from_dict(follower_v2_VehSpeed_dict, orient = 'index').rename(columns = {0:'follower_v2_VehSpeed'}).rename_axis("Time").reset_index()
    follower_v2_VehSpeed_df['DateTime'] = pd.to_datetime(follower_v2_VehSpeed_df['Time'],unit='s')

    # call for lead_v1 vehicle speed
    lead_v1_VehSpeed_dict = {}
    for topic, msg, t in lead_v1_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=lead_v1_engaged_time):
        lead_v1_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x # .twist.linear.x is the vehicle's longitudinal velocity in m/s 
    lead_v1_VehSpeed_df = pd.DataFrame.from_dict(lead_v1_VehSpeed_dict, orient = 'index').rename(columns = {0:'lead_v1_VehSpeed'}).rename_axis("Time").reset_index()
    lead_v1_VehSpeed_df['DateTime'] = pd.to_datetime(lead_v1_VehSpeed_df['Time'],unit='s')

    # call for actual gap
    follower_v2_ActualGap_dict = {}
    for topic, msg, t in follower_v2_bag.read_messages(topics = ["/guidance/platooning_info"], start_time=follower_v2_engaged_time):
        follower_v2_ActualGap_dict[t.to_sec()]=msg.actual_gap
    follower_v2_ActualGap_df = pd.DataFrame.from_dict(follower_v2_ActualGap_dict, orient='index').rename(columns={0:'follower_v2_ActualGap'}).rename_axis("Time").reset_index()
    follower_v2_ActualGap_df['DateTime'] = pd.to_datetime(follower_v2_ActualGap_df['Time'],unit='s')

    follower_v3_ActualGap_dict = {}
    for topic, msg, t in follower_v3_bag.read_messages(topics = ["/guidance/platooning_info"], start_time=follower_v3_engaged_time):
        follower_v3_ActualGap_dict[t.to_sec()]=msg.actual_gap
    follower_v3_ActualGap_df = pd.DataFrame.from_dict(follower_v3_ActualGap_dict, orient='index').rename(columns={0:'follower_v3_ActualGap'}).rename_axis("Time").reset_index()
    follower_v3_ActualGap_df['DateTime'] = pd.to_datetime(follower_v3_ActualGap_df['Time'],unit='s')

    # Obtain timestamp at which each vehicle first requests to join platoon
    follower_v2_RequestPlatoon_dict = {}
    for topic, msg, t in follower_v2_bag.read_messages(topics = ["/message/outgoing_mobility_request"]):
        follower_v2_RequestPlatoon_dict[t.to_sec()] = msg.m_header.timestamp  
    follower_v2_RequestPlatoon_df = pd.DataFrame.from_dict(follower_v2_RequestPlatoon_dict, orient = 'index').rename(columns = {0:'follower_v2_RequestTime'}).rename_axis("Time").reset_index()
    follower_v2_RequestPlatoon_df['DateTime'] = pd.to_datetime(follower_v2_RequestPlatoon_df['Time'],unit='s')
    # first instance of an “outgoing mobility request” 
    follower_v2_RequestPlatooning = (follower_v2_RequestPlatoon_df['DateTime'].iloc[0])

    follower_v3_RequestPlatoon_dict = {}
    for topic, msg, t in follower_v3_bag.read_messages(topics = ["/message/outgoing_mobility_request"]):
        follower_v3_RequestPlatoon_dict[t.to_sec()] = msg.m_header.timestamp  
    follower_v3_RequestPlatoon_df = pd.DataFrame.from_dict(follower_v3_RequestPlatoon_dict, orient = 'index').rename(columns = {0:'follower_v3_RequestTime'}).rename_axis("Time").reset_index()
    follower_v3_RequestPlatoon_df['DateTime'] = pd.to_datetime(follower_v3_RequestPlatoon_df['Time'],unit='s')
    # first instance of an “outgoing mobility request” 
    follower_v3_RequestPlatooning = (follower_v3_RequestPlatoon_df['DateTime'].iloc[0])

    lead_v1_RequestPlatoon_dict = {}
    for topic, msg, t in lead_v1_bag.read_messages(topics = ["/message/outgoing_mobility_request"]):
        lead_v1_RequestPlatoon_dict[t.to_sec()] = msg.m_header.timestamp  
    lead_v1_RequestPlatoon_df = pd.DataFrame.from_dict(lead_v1_RequestPlatoon_dict, orient = 'index').rename(columns = {0:'lead_v1_RequestTime'}).rename_axis("Time").reset_index()
    lead_v1_RequestPlatoon_df['DateTime'] = pd.to_datetime(lead_v1_RequestPlatoon_df['Time'],unit='s')
    # first instance of an “outgoing mobility request” 
    if lead_v1_RequestPlatoon_dict != {}:
        lead_v1_RequestPlatooning = (lead_v1_RequestPlatoon_df['DateTime'].iloc[0])

    # call for advisory speed
    follower_v2_AdvisorySpeed_dict = {}
    for topic, msg, t in follower_v2_bag.read_messages(topics = ["/environment/active_geofence"], start_time=follower_v2_engaged_time):
        follower_v2_AdvisorySpeed_dict[t.to_sec()]=msg.advisory_speed
    follower_v2_AdvisorySpeed_df = pd.DataFrame.from_dict(follower_v2_AdvisorySpeed_dict, orient='index').rename(columns={0:'follower_v2_AdvisorySpeed'}).rename_axis("Time").reset_index()
    follower_v2_AdvisorySpeed_df['DateTime'] = pd.to_datetime(follower_v2_AdvisorySpeed_df['Time'],unit='s')

    follower_v2_StartPlatooning_time = pd.to_datetime(follower_v2_platoon_flag.to_sec(),unit='s')
    follower_v3_StartPlatooning_time = pd.to_datetime(follower_v3_platoon_flag.to_sec(),unit='s')
    lead_v1_StartLeadPlatooning_time = pd.to_datetime(lead_v1_platoon_flag.to_sec(),unit='s')

    # Generate graph
    fig = make_subplots(specs=[[{"secondary_y":True}]])

    fig.add_trace(go.Scatter(x=lead_v1_VehSpeed_df['DateTime'], y=lead_v1_VehSpeed_df['lead_v1_VehSpeed'], line=dict(color = 'blue'), name = lead_v1_name + "_VehSpeed"))
    if lead_v1_RequestPlatoon_dict != {}:
        fig.add_trace(go.Scatter(x=[lead_v1_RequestPlatooning], y=[0],line=dict(color = 'lightblue', dash = 'dash'),name = lead_v1_name + "_RequestPlatooning"),secondary_y=False)
    fig.add_trace(go.Scatter(x=[lead_v1_StartLeadPlatooning_time], y=[0], line=dict(color = 'darkblue', dash = 'dash'), name = lead_v1_name + "_StartLeadPlatooning"), secondary_y=False)

    fig.add_trace(go.Scatter(x=follower_v2_VehSpeed_df['DateTime'], y=follower_v2_VehSpeed_df['follower_v2_VehSpeed'],line = dict(color='red'), name = follower_v2_name + "_VehSpeed"))
    fig.add_trace(go.Scatter(x=follower_v2_ActualGap_df['DateTime'], y=follower_v2_ActualGap_df['follower_v2_ActualGap'],line = dict(color='red', width=2, dash='dash'), name = follower_v2_name + "_FollowingGap"), secondary_y=True)
    fig.add_trace(go.Scatter(x=[follower_v2_StartPlatooning_time], y=[0], line=dict(color = 'darkred', dash = 'dash'), name = follower_v2_name + "_StartPlatooning"), secondary_y=False)

    fig.add_trace(go.Scatter(x=follower_v3_VehSpeed_df['DateTime'], y=follower_v3_VehSpeed_df['follower_v3_VehSpeed'],line = dict(color='green'), name = follower_v3_name + "_VehSpeed"))
    fig.add_trace(go.Scatter(x=follower_v3_ActualGap_df['DateTime'], y=follower_v3_ActualGap_df['follower_v3_ActualGap'],line = dict(color='green', width=2, dash='dash'), name = follower_v3_name + "_FollowingGap"), secondary_y=True)
    fig.add_trace(go.Scatter(x=[follower_v3_RequestPlatooning], y=[0],line=dict(color = 'lightgreen', dash = 'dash'),name = follower_v3_name + "_RequestPlatooning"),secondary_y=False)
    fig.add_trace(go.Scatter(x=[follower_v3_StartPlatooning_time], y=[0], line=dict(color = 'darkgreen', dash = 'dash'), name = follower_v3_name + "_StartPlatooning"), secondary_y=False)

    fig.add_trace(go.Scatter(x=follower_v2_AdvisorySpeed_df['DateTime'], y=follower_v2_AdvisorySpeed_df['follower_v2_AdvisorySpeed'],line = dict(color='orange', width=2, dash='dash'), name = "SpeedHarmonizationZone"), secondary_y=False)

    fig.update_layout(title= test_number + "_" + run_number + "_" + lead_v1_name + "_" + follower_v2_name + "_" + follower_v3_name + "- 3 Vehicles")
    fig.update_xaxes(title="Time (s)")
    fig.update_yaxes(title="Speed (m/s)")
    fig.update_yaxes(title="Gap (m)", secondary_y=True)
    fig.update_yaxes(range=[0, 120], secondary_y=True)

    fig.show()
    # Save graph to file
    fig.write_html(file_save + test_number + "_" + run_number + "_" + lead_v1_name + "_" + follower_v2_name + "_" + follower_v3_name + "_3_Vehicle_Speed_Gap_Graph_1.html")

In [ ]:
# 3 vehicle test run
if three_vehicle_bool:

    # call for follower_v3 vehicle speed
    follower_v3_VehSpeed_dict = {}
    for topic, msg, t in follower_v3_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=follower_v3_engaged_time):
        follower_v3_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x  # .twist.linear.x is the vehicle's longitudinal velocity in m/s
    follower_v3_VehSpeed_df = pd.DataFrame.from_dict(follower_v3_VehSpeed_dict, orient = 'index').rename(columns = {0:'follower_v3_VehSpeed'}).rename_axis("Time").reset_index()
    follower_v3_VehSpeed_df['DateTime'] = pd.to_datetime(follower_v3_VehSpeed_df['Time'],unit='s')

    # call for follower_v2 vehicle speed
    follower_v2_VehSpeed_dict = {}
    for topic, msg, t in follower_v2_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=follower_v2_engaged_time):
        follower_v2_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x  # .twist.linear.x is the vehicle's longitudinal velocity in m/s
    follower_v2_VehSpeed_df = pd.DataFrame.from_dict(follower_v2_VehSpeed_dict, orient = 'index').rename(columns = {0:'follower_v2_VehSpeed'}).rename_axis("Time").reset_index()
    follower_v2_VehSpeed_df['DateTime'] = pd.to_datetime(follower_v2_VehSpeed_df['Time'],unit='s')

    # call for lead_v1 vehicle speed
    lead_v1_VehSpeed_dict = {}
    for topic, msg, t in lead_v1_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=lead_v1_engaged_time):
        lead_v1_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x  # .twist.linear.x is the vehicle's longitudinal velocity in m/s
    lead_v1_VehSpeed_df = pd.DataFrame.from_dict(lead_v1_VehSpeed_dict, orient = 'index').rename(columns = {0:'lead_v1_VehSpeed'}).rename_axis("Time").reset_index()
    lead_v1_VehSpeed_df['DateTime'] = pd.to_datetime(lead_v1_VehSpeed_df['Time'],unit='s')

    # call for lead_v1 cross track
    lead_v1_CrossTrack_dict = {}
    for topic, msg, t in lead_v1_bag.read_messages(topics = ["/guidance/route_state"], start_time=lead_v1_engaged_time):
        lead_v1_CrossTrack_dict[t.to_sec()] = msg.cross_track
    lead_v1_CrossTrack_df = pd.DataFrame.from_dict(lead_v1_CrossTrack_dict, orient = 'index').rename(columns = {0:'lead_v1_CrossTrack'}).rename_axis("Time").reset_index()
    lead_v1_CrossTrack_df['DateTime'] = pd.to_datetime(lead_v1_CrossTrack_df['Time'],unit='s')


    lead_v1_RequestPlatoon_dict = {}
    for topic, msg, t in lead_v1_bag.read_messages(topics = ["/message/outgoing_mobility_request"]):
        lead_v1_RequestPlatoon_dict[t.to_sec()] = msg.m_header.timestamp  
    lead_v1_RequestPlatoon_df = pd.DataFrame.from_dict(lead_v1_RequestPlatoon_dict, orient = 'index').rename(columns = {0:'lead_v1_RequestTime'}).rename_axis("Time").reset_index()
    lead_v1_RequestPlatoon_df['DateTime'] = pd.to_datetime(lead_v1_RequestPlatoon_df['Time'],unit='s')
    # first instance of an “outgoing mobility request” 
    if lead_v1_RequestPlatoon_dict != {}:
        lead_v1_RequestPlatooning = (lead_v1_RequestPlatoon_df['DateTime'].iloc[0])

    follower_v2_StartPlatooning_time = pd.to_datetime(follower_v2_platoon_flag.to_sec(),unit='s')
    follower_v3_StartPlatooning_time = pd.to_datetime(follower_v3_platoon_flag.to_sec(),unit='s')
    lead_v1_StartLeadPlatooning_time = pd.to_datetime(lead_v1_platoon_flag.to_sec(),unit='s')

    # Generate graph
    fig = make_subplots(specs=[[{"secondary_y":True}]])

    fig.add_trace(go.Scatter(x=follower_v2_VehSpeed_df['DateTime'], y=follower_v2_VehSpeed_df['follower_v2_VehSpeed'], name = follower_v2_name + "_VehSpeed"))
    fig.add_trace(go.Scatter(x=follower_v3_VehSpeed_df['DateTime'], y=follower_v3_VehSpeed_df['follower_v3_VehSpeed'], name = follower_v3_name + "_VehSpeed"))
    fig.add_trace(go.Scatter(x=lead_v1_VehSpeed_df['DateTime'], y=lead_v1_VehSpeed_df['lead_v1_VehSpeed'], name = lead_v1_name + "_VehSpeed"))

    fig.add_trace(go.Scatter(x=lead_v1_CrossTrack_df['DateTime'], y=lead_v1_CrossTrack_df['lead_v1_CrossTrack'],line = dict(dash='dash'), name = lead_v1_name + "_CrossTrack"), secondary_y=True)

    fig.add_trace(go.Scatter(x=[follower_v3_StartPlatooning_time], y=[0], line=dict(color = 'blue', dash = 'dash'), name = follower_v3_name + "_StartPlatooning"), secondary_y=False)

    if lead_v1_RequestPlatoon_dict != {}:
        fig.add_trace(go.Scatter(x=[lead_v1_RequestPlatooning], y=[0],line=dict(color = 'gold', dash = 'dash'),name = lead_v1_name + "_RequestPlatooning"),secondary_y=False)
    fig.add_trace(go.Scatter(x=[lead_v1_StartLeadPlatooning_time], y=[0], line=dict(color = 'red', dash = 'dash'), name = lead_v1_name + "_StartLeadPlatooning"), secondary_y=False)

    fig.update_layout(title= test_number + "_" + run_number + "_" + lead_v1_name + "_" + follower_v2_name + "_" + follower_v3_name + "- 3 Vehicles")
    fig.update_xaxes(title="Time (s)")
    fig.update_yaxes(title="Speed (m/s)")
    fig.update_yaxes(title="Cross Track (m)", secondary_y=True)
    fig.update_yaxes(range=[-2.0, 2.0], secondary_y=True)

    fig.show()
    # Save graph to file
    fig.write_html(file_save + test_number + "_" + run_number + "_" + lead_v1_name + "_" + follower_v2_name + "_" + follower_v3_name + "_3_Vehicle_Speed_Gap_Graph_2.html")

In [ ]:
# Use this section for 2 vehicle test run
if two_vehicle_bool:

      # extract timestamp for when CARMA platform is engaged
      # msg.state == 4 indicates that the CARMA platform system is engaged

      lead_v1b_engaged_time = 0
      for topic, msg, t in lead_v1b_bag.read_messages(topics = ["/guidance/state"]):
            if msg.state == 4:
                  lead_v1b_engaged_time = t
                  break
      follower_v2b_engaged_time = 0
      for topic, msg, t in follower_v2b_bag.read_messages(topics = ["/guidance/state"]):
            if msg.state == 4:
                  follower_v2b_engaged_time = t
                  break

      platoon_flag = 0
      for topic, msg, t in lead_v1b_bag.read_messages(topics = ["/guidance/platoon_info"]):
            if msg.state == 4:
                  platoon_flag = t
                  break

In [ ]:
# Use this section for 2 vehicle test run
if two_vehicle_bool:

    # call for lead_v1b vehicle speed
    lead_v1b_VehSpeed_dict = {}
    for topic, msg, t in lead_v1b_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=lead_v1b_engaged_time):
        lead_v1b_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x  # .twist.linear.x is the vehicle's longitudinal velocity in m/s
    lead_v1b_VehSpeed_df = pd.DataFrame.from_dict(lead_v1b_VehSpeed_dict, orient = 'index').rename(columns = {0:'lead_v1b_VehSpeed'}).rename_axis("Time").reset_index()
    lead_v1b_VehSpeed_df['DateTime'] = pd.to_datetime(lead_v1b_VehSpeed_df['Time'],unit='s')

    # call for follower_v2b vehicle speed
    follower_v2b_VehSpeed_dict = {}
    for topic, msg, t in follower_v2b_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=follower_v2b_engaged_time):
        follower_v2b_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x  # .twist.linear.x is the vehicle's longitudinal velocity in m/s
    follower_v2b_VehSpeed_df = pd.DataFrame.from_dict(follower_v2b_VehSpeed_dict, orient = 'index').rename(columns = {0:'follower_v2b_VehSpeed'}).rename_axis("Time").reset_index()
    follower_v2b_VehSpeed_df['DateTime'] = pd.to_datetime(follower_v2b_VehSpeed_df['Time'],unit='s')

    ActualGap_dict = {}
    for topic, msg, t in follower_v2b_bag.read_messages(topics = ["/guidance/platooning_info"], start_time=follower_v2b_engaged_time):
        ActualGap_dict[t.to_sec()]=msg.actual_gap
    ActualGap_df = pd.DataFrame.from_dict(ActualGap_dict, orient='index').rename(columns={0:'ActualGap'}).rename_axis("Time").reset_index()
    ActualGap_df['DateTime'] = pd.to_datetime(ActualGap_df['Time'],unit='s')

    RequestPlatoon_dict = {}
    for topic, msg, t in follower_v2b_bag.read_messages(topics = ["/message/outgoing_mobility_request"]):
        RequestPlatoon_dict[t.to_sec()] = msg.m_header.timestamp  
    RequestPlatoon_df = pd.DataFrame.from_dict(RequestPlatoon_dict, orient = 'index').rename(columns = {0:'RequestTime'}).rename_axis("Time").reset_index()
    RequestPlatoon_df['DateTime'] = pd.to_datetime(RequestPlatoon_df['Time'],unit='s')
    # first instance of an “outgoing mobility request” 
    RequestPlatooning = (RequestPlatoon_df['DateTime'].iloc[0])

    StartPlatooning_time = pd.to_datetime(platoon_flag.to_sec(),unit='s')

    # Generate graph
    fig = make_subplots(specs=[[{"secondary_y":True}]])

    fig.add_trace(go.Scatter(x=lead_v1b_VehSpeed_df['DateTime'], y=lead_v1b_VehSpeed_df['lead_v1b_VehSpeed'], name = lead_v1b_name + "_VehSpeed"))
    fig.add_trace(go.Scatter(x=follower_v2b_VehSpeed_df['DateTime'], y=follower_v2b_VehSpeed_df['follower_v2b_VehSpeed'], name = follower_v2b_name + "_VehSpeed"))
    fig.add_trace(go.Scatter(x=ActualGap_df['DateTime'], y=ActualGap_df['ActualGap'], name = "FollowingGap"), secondary_y=True)

    fig.add_trace(go.Scatter(x=[RequestPlatooning], y=[0],line=dict(color = 'orange', dash = 'dash'),name ="RequestPlatooning"),secondary_y=False)
    fig.add_trace(go.Scatter(x=[StartPlatooning_time], y=[0], line=dict(color = 'green', dash = 'dash'), name ="StartPlatooning"), secondary_y=False),

    fig.add_vline(x=RequestPlatooning, line_width=2, line_dash="dash", line_color="orange")
    fig.add_vline(x=StartPlatooning_time, line_width=2, line_dash="dash", line_color="green")

    fig.update_layout(title= test_number_b + "_" + run_number_b + "_" + lead_v1b_name + "_" + follower_v2b_name + " - 2 Vehicles")
    fig.update_xaxes(title="Local Time (s)")
    fig.update_yaxes(title="Speed (m/s)")
    fig.update_yaxes(title="ActualGap (m)", secondary_y=True)

    fig.show()
    # Save graph to file
    fig.write_html(file_b_save + test_number_b + "_" + run_number_b + "_" + lead_v1b_name + "_" + follower_v2b_name + "_" + "_2_Vehicle_Speed_Gap_Graph_1.html")

In [ ]:
# Use this section for 2 vehicle test run
if two_vehicle_bool:

    # call for lead_v1b vehicle speed
    lead_v1b_VehSpeed_dict = {}
    for topic, msg, t in lead_v1b_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=lead_v1b_engaged_time):
        lead_v1b_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x  # .twist.linear.x is the vehicle's longitudinal velocity in m/s
    lead_v1b_VehSpeed_df = pd.DataFrame.from_dict(lead_v1b_VehSpeed_dict, orient = 'index').rename(columns = {0:'lead_v1b_VehSpeed'}).rename_axis("Time").reset_index()

    # call for follower_v2b vehicle speed
    follower_v2b_VehSpeed_dict = {}
    for topic, msg, t in follower_v2b_bag.read_messages(topics = ["/hardware_interface/vehicle/twist"], start_time=follower_v2b_engaged_time):
        follower_v2b_VehSpeed_dict[t.to_sec()] = msg.twist.linear.x  # .twist.linear.x is the vehicle's longitudinal velocity in m/s
    follower_v2b_VehSpeed_df = pd.DataFrame.from_dict(follower_v2b_VehSpeed_dict, orient = 'index').rename(columns = {0:'follower_v2b_VehSpeed'}).rename_axis("Time").reset_index()

    # generating the dataframe from our dictionary
    # renaming the standard dataframe column headers to timestamp, x, and y

    # Save csv file
    follower_v2b_VehSpeed_df.to_csv(file_b_save + test_number_b + "_" + run_number_b + "_" + follower_v2b_name + "_2_Vehicle_Speed.csv")

    lead_v1b_CarmaEngaged = lead_v1b_engaged_time.to_sec()
    follower_v2b_CarmaEngaged = follower_v2b_engaged_time.to_sec()

    lead_v1b_VehSpeed_timeArray = next(iter((lead_v1b_VehSpeed_df.items())))[1]
    lead_v1b_VehSpeed_df['ElapseTime'] = lead_v1b_VehSpeed_timeArray-lead_v1b_CarmaEngaged
    follower_v2b_VehSpeed_timeArray = next(iter((follower_v2b_VehSpeed_df.items())))[1]
    follower_v2b_VehSpeed_df['ElapseTime'] = follower_v2b_VehSpeed_timeArray-follower_v2b_CarmaEngaged

    # Generate graph
    fig = make_subplots(specs=[[{"secondary_y":True}]])
    fig.add_trace(go.Scatter(x=lead_v1b_VehSpeed_df['ElapseTime'], y=lead_v1b_VehSpeed_df['lead_v1b_VehSpeed'], name = lead_v1b_name + "_VehSpeed"))
    fig.add_trace(go.Scatter(x=follower_v2b_VehSpeed_df['ElapseTime'], y=follower_v2b_VehSpeed_df['follower_v2b_VehSpeed'], name = follower_v2b_name + "_VehSpeed"))

    fig.update_layout(title= test_number_b + "_" + run_number_b + "_" + lead_v1b_name + "_" + follower_v2b_name + " - 2 Vehicles")
    fig.update_xaxes(title="Time (s)")
    fig.update_yaxes(title="Speed (m/s)")

    fig.show()
    # Save graph to file
    fig.write_html(file_b_save + test_number_b + "_" + run_number_b + "_" + lead_v1b_name + "_" + follower_v2b_name + "_" + "_2_Vehicle_Speed_Gap_Graph_2.html")